In [0]:
# Importar bibliotecas necessárias
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurar estilo dos gráficos
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Definir paletas de cores
paleta_sintomas = px.colors.qualitative.Pastel
paleta_regioes = px.colors.qualitative.Bold
paleta_faixas = px.colors.sequential.Viridis

# Carregar os dados da tabela de resultados
resultados_df = spark.table("covid_analise_resultados")
display(resultados_df)

# Carregar os dados da tabela principal para gerar gráficos detalhados
df_covid = spark.table("covid_data_tratado")

# Converter para pandas para facilitar a visualização
df_pd = df_covid.toPandas()

# Configurar o notebook para exibir gráficos plotly
displayHTML("Análise Gráfica dos Resultados PNAD-COVID-19")

In [0]:
# ===== 1. GRÁFICO DE PREVALÊNCIA DE SINTOMAS =====
# Identificar colunas de sintomas
sintomas_cols = [col for col in df_pd.columns if col.startswith("SE_TEVE_")]

# Calcular prevalência de cada sintoma
sintomas_prevalencia = {}
for col in sintomas_cols:
    sintoma_nome = col.replace("SE_TEVE_", "").replace("_", " ").title()
    sintomas_prevalencia[sintoma_nome] = (df_pd[col] == "Sim").mean() * 100

# Criar DataFrame para o gráfico
sintomas_df = pd.DataFrame({
    'Sintoma': list(sintomas_prevalencia.keys()),
    'Prevalência (%)': list(sintomas_prevalencia.values())
}).sort_values('Prevalência (%)', ascending=False)

# Criar gráfico de barras horizontais
fig_sintomas = px.bar(
    sintomas_df,
    y='Sintoma',
    x='Prevalência (%)',
    orientation='h',
    title='Prevalência de Sintomas na População',
    color='Prevalência (%)',
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={'Prevalência (%)': 'Percentual de Pessoas (%)'}
)

fig_sintomas.update_layout(height=500, width=800)
display(fig_sintomas)

In [0]:
# ===== 2. GRÁFICO DE SINTOMAS GRAVES POR FAIXA ETÁRIA =====
# Calcular percentual de sintomas graves por faixa etária
sintomas_graves_faixa = df_covid.groupBy("FAIXA_ETARIA") \
                        .agg(
                            F.count("*").alias("total"),
                            F.sum(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 1).otherwise(0)).alias("casos_graves")
                        ) \
                        .withColumn("percentual", F.col("casos_graves") / F.col("total") * 100) \
                        .orderBy("percentual", ascending=False)

# Converter para pandas
sintomas_graves_pd = sintomas_graves_faixa.toPandas()

# Criar gráfico de barras
fig_faixas = px.bar(
    sintomas_graves_pd,
    x='FAIXA_ETARIA',
    y='percentual',
    title='Percentual de Sintomas Graves por Faixa Etária',
    color='percentual',
    color_continuous_scale=px.colors.sequential.Reds,
    labels={
        'FAIXA_ETARIA': 'Faixa Etária',
        'percentual': 'Percentual com Sintomas Graves (%)'
    }
)

fig_faixas.update_layout(height=500, width=800)
display(fig_faixas)

In [0]:
# ===== 3. MAPA DE CALOR: NÍVEL DE RISCO POR REGIÃO =====
# Calcular distribuição de níveis de risco por região
risco_regiao = df_covid.groupBy("REGIAO", "NIVEL_RISCO") \
              .count() \
              .withColumn("percentual", F.col("count") / F.sum("count").over(Window.partitionBy("REGIAO")) * 100)

# Converter para pandas e pivotar para formato de matriz
risco_regiao_pd = risco_regiao.toPandas()
risco_pivot = risco_regiao_pd.pivot(index='REGIAO', columns='NIVEL_RISCO', values='percentual').fillna(0)

# Garantir ordem correta das colunas
ordem_risco = ['Baixo', 'Médio', 'Alto', 'Muito Alto']
for nivel in ordem_risco:
    if nivel not in risco_pivot.columns:
        risco_pivot[nivel] = 0
risco_pivot = risco_pivot[ordem_risco]

# Criar mapa de calor
fig_heatmap = px.imshow(
    risco_pivot,
    labels=dict(x="Nível de Risco", y="Região", color="Percentual (%)"),
    x=ordem_risco,
    y=risco_pivot.index,
    color_continuous_scale=px.colors.sequential.Reds,
    title="Distribuição de Níveis de Risco por Região"
)

fig_heatmap.update_layout(height=500, width=800)
display(fig_heatmap)

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define the window specification
window_spec = Window.partitionBy()

# Calcular distribuição de auxílio emergencial por região
auxilio_regiao = df_covid.filter(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim") \
                .groupBy("REGIAO") \
                .count() \
                .withColumn("percentual", F.col("count") / F.sum("count").over(window_spec) * 100)

# Converter para pandas
auxilio_pd = auxilio_regiao.toPandas()

# Criar gráfico de pizza
fig_auxilio = px.pie(
    auxilio_pd,
    values='percentual',
    names='REGIAO',
    title='Distribuição de Auxílio Emergencial por Região',
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig_auxilio.update_layout(height=500, width=800)
display(fig_auxilio)

In [0]:
# ===== 5. GRÁFICO DE BARRAS EMPILHADAS: TRABALHO REMOTO POR ESCOLARIDADE =====
# Calcular distribuição de trabalho remoto por escolaridade
if "TRABALHO_REMOTO" in df_covid.columns and "ESCOLARIDADE" in df_covid.columns:
    trabalho_escolaridade = df_covid.groupBy("ESCOLARIDADE") \
                           .pivot("TRABALHO_REMOTO") \
                           .count() \
                           .fillna(0)

    # Converter para pandas
    trabalho_pd = trabalho_escolaridade.toPandas()

    # Calcular percentuais
    for col in trabalho_pd.columns:
        if col != "ESCOLARIDADE":
            trabalho_pd[f"{col}_pct"] = trabalho_pd[col] / trabalho_pd.sum(axis=1) * 100

    # Criar gráfico de barras empilhadas
    fig_trabalho = go.Figure()

    # Adicionar barras para cada categoria
    if "Sim" in trabalho_pd.columns:
        fig_trabalho.add_trace(go.Bar(
            x=trabalho_pd["ESCOLARIDADE"],
            y=trabalho_pd["Sim_pct"] if "Sim_pct" in trabalho_pd.columns else trabalho_pd["Sim"],
            name="Trabalho Remoto",
            marker_color='rgb(55, 83, 109)'
        ))

    if "Não" in trabalho_pd.columns:
        fig_trabalho.add_trace(go.Bar(
            x=trabalho_pd["ESCOLARIDADE"],
            y=trabalho_pd["Não_pct"] if "Não_pct" in trabalho_pd.columns else trabalho_pd["Não"],
            name="Trabalho Presencial",
            marker_color='rgb(26, 118, 255)'
        ))

    fig_trabalho.update_layout(
        title='Distribuição de Trabalho Remoto por Escolaridade',
        xaxis_title='Nível de Escolaridade',
        yaxis_title='Percentual (%)',
        barmode='stack',
        height=500,
        width=900
    )

    display(fig_trabalho)

In [0]:
# ===== 6. GRÁFICO DE LINHA: EVOLUÇÃO TEMPORAL DE SINTOMAS =====
# Verificar se temos dados temporais suficientes
if "MES_PESQUISA" in df_covid.columns:
    # Calcular prevalência de sintomas graves por mês
    evolucao_sintomas = df_covid.groupBy("MES_PESQUISA") \
                       .agg(
                           F.count("*").alias("total"),
                           F.sum(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 1).otherwise(0)).alias("casos_graves")
                       ) \
                       .withColumn("percentual", F.col("casos_graves") / F.col("total") * 100) \
                       .orderBy("MES_PESQUISA")

    # Converter para pandas
    evolucao_pd = evolucao_sintomas.toPandas()

    # Criar gráfico de linha
    fig_evolucao = px.line(
        evolucao_pd,
        x="MES_PESQUISA",
        y="percentual",
        markers=True,
        title="Evolução de Sintomas Graves ao Longo do Tempo",
        labels={
            "MES_PESQUISA": "Mês da Pesquisa",
            "percentual": "Percentual com Sintomas Graves (%)"
        }
    )

    fig_evolucao.update_layout(height=500, width=800)
    display(fig_evolucao)

In [0]:
# ===== 7. GRÁFICO DE RADAR: PERFIL DE VULNERABILIDADE =====
# Criar perfil de vulnerabilidade por região
vulnerabilidade = df_covid.filter(F.col("VULNERAVEL") == "Sim") \
                 .groupBy("REGIAO") \
                 .agg(
                     F.count("*").alias("total_vulneraveis"),
                     F.avg(F.when(F.col("TEM_PLANO_SAUDE") == "Não", 100).otherwise(0)).alias("sem_plano"),
                     F.avg(F.when(F.col("FAIXA_ETARIA") == "Idoso", 100).otherwise(0)).alias("idosos"),
                     F.avg(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 100).otherwise(0)).alias("sintomas_graves"),
                     F.avg(F.when(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim", 100).otherwise(0)).alias("recebe_auxilio")
                 )

# Converter para pandas
vulnerabilidade_pd = vulnerabilidade.toPandas()

# Criar gráfico de radar
fig_radar = go.Figure()

# Adicionar uma linha para cada região
for i, regiao in enumerate(vulnerabilidade_pd["REGIAO"]):
    fig_radar.add_trace(go.Scatterpolar(
        r=[
            vulnerabilidade_pd.loc[i, "sem_plano"],
            vulnerabilidade_pd.loc[i, "idosos"],
            vulnerabilidade_pd.loc[i, "sintomas_graves"],
            vulnerabilidade_pd.loc[i, "recebe_auxilio"],
            vulnerabilidade_pd.loc[i, "sem_plano"]  # Repetir o primeiro para fechar o polígono
        ],
        theta=["Sem Plano de Saúde", "Idosos", "Sintomas Graves", "Recebe Auxílio", "Sem Plano de Saúde"],
        name=regiao,
        fill='toself'
    ))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    title="Perfil de Vulnerabilidade por Região",
    height=600,
    width=800
)

display(fig_radar)

In [0]:
# Importar bibliotecas necessárias
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql.window import Window

# Configurar estilo dos gráficos
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Definir paletas de cores
paleta_sintomas = px.colors.qualitative.Pastel
paleta_regioes = px.colors.qualitative.Bold
paleta_faixas = px.colors.sequential.Viridis

# Carregar os dados da tabela principal
df_covid = spark.table("covid_data_tratado")

# Converter para pandas para facilitar a visualização
df_pd = df_covid.toPandas()

# ===== PREPARAR DADOS PARA O DASHBOARD =====

# 1. Dados para gráfico de sintomas graves por faixa etária
sintomas_graves_faixa = df_covid.groupBy("FAIXA_ETARIA") \
                        .agg(
                            F.count("*").alias("total"),
                            F.sum(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 1).otherwise(0)).alias("casos_graves")
                        ) \
                        .withColumn("percentual", F.col("casos_graves") / F.col("total") * 100) \
                        .orderBy("percentual", ascending=False)

sintomas_graves_pd = sintomas_graves_faixa.toPandas()

# 2. Dados para gráfico de auxílio emergencial por região
window_spec = Window.partitionBy()
auxilio_regiao = df_covid.filter(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim") \
                .groupBy("REGIAO") \
                .count() \
                .withColumn("percentual", F.col("count") / F.sum("count").over(window_spec) * 100)

auxilio_pd = auxilio_regiao.toPandas()

# 3. Dados para mapa de calor de níveis de risco por região
risco_regiao = df_covid.groupBy("REGIAO", "NIVEL_RISCO") \
              .count() \
              .withColumn("percentual", F.col("count") / F.sum("count").over(Window.partitionBy("REGIAO")) * 100)

risco_regiao_pd = risco_regiao.toPandas()

# Garantir que temos todas as combinações de região e nível de risco
regioes = risco_regiao_pd["REGIAO"].unique()
niveis = ["Baixo", "Médio", "Alto", "Muito Alto"]

# Criar DataFrame completo com todas as combinações
risco_completo = []
for regiao in regioes:
    for nivel in niveis:
        # Verificar se a combinação existe
        match = risco_regiao_pd[(risco_regiao_pd["REGIAO"] == regiao) & 
                               (risco_regiao_pd["NIVEL_RISCO"] == nivel)]
        if len(match) > 0:
            risco_completo.append({
                "REGIAO": regiao,
                "NIVEL_RISCO": nivel,
                "percentual": match["percentual"].values[0]
            })
        else:
            risco_completo.append({
                "REGIAO": regiao,
                "NIVEL_RISCO": nivel,
                "percentual": 0
            })

risco_completo_df = pd.DataFrame(risco_completo)

# Pivotar para formato de matriz para o mapa de calor
risco_pivot = risco_completo_df.pivot(index='REGIAO', columns='NIVEL_RISCO', values='percentual').fillna(0)

# Garantir ordem correta das colunas
for nivel in niveis:
    if nivel not in risco_pivot.columns:
        risco_pivot[nivel] = 0
risco_pivot = risco_pivot[niveis]

# 4. Dados para gráfico de radar de vulnerabilidade
vulnerabilidade = df_covid.groupBy("REGIAO") \
                 .agg(
                     F.avg(F.when(F.col("TEM_PLANO_SAUDE") == "Não", 100).otherwise(0)).alias("sem_plano"),
                     F.avg(F.when(F.col("FAIXA_ETARIA") == "Idoso", 100).otherwise(0)).alias("idosos"),
                     F.avg(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 100).otherwise(0)).alias("sintomas_graves"),
                     F.avg(F.when(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim", 100).otherwise(0)).alias("recebe_auxilio")
                 )

vulnerabilidade_pd = vulnerabilidade.toPandas()

# ===== CRIAR DASHBOARD MELHORADO =====
# Criar um dashboard com os principais indicadores com mais espaço entre os gráficos
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Prevalência de Sintomas Graves por Faixa Etária",
        "Distribuição de Auxílio Emergencial",
        "Níveis de Risco por Região",
        "Perfil de Vulnerabilidade"
    ),
    specs=[
        [{"type": "bar"}, {"type": "pie"}],
        [{"type": "heatmap"}, {"type": "polar"}]
    ],
    vertical_spacing=0.15,  # Aumentar espaçamento vertical
    horizontal_spacing=0.1  # Aumentar espaçamento horizontal
)

# 1. Gráfico de barras (sintomas por faixa etária)
# Usar um único trace para todas as barras
fig_dashboard.add_trace(
    go.Bar(
        x=sintomas_graves_pd["FAIXA_ETARIA"],
        y=sintomas_graves_pd["percentual"],
        marker_color=paleta_faixas[:len(sintomas_graves_pd)],
        showlegend=False
    ),
    row=1, col=1
)

# 2. Gráfico de pizza (auxílio emergencial)
fig_dashboard.add_trace(
    go.Pie(
        labels=auxilio_pd["REGIAO"],
        values=auxilio_pd["percentual"],
        marker_colors=paleta_regioes[:len(auxilio_pd)],
        textinfo='label+percent',
        showlegend=False
    ),
    row=1, col=2
)

# 3. Mapa de calor (níveis de risco)
fig_dashboard.add_trace(
    go.Heatmap(
        z=risco_pivot.values,
        x=risco_pivot.columns,
        y=risco_pivot.index,
        colorscale="Reds",
        showscale=True,
        showlegend=False
    ),
    row=2, col=1
)

# 4. Gráfico de radar (vulnerabilidade)
# Usar cores distintas para cada região
for i, regiao in enumerate(vulnerabilidade_pd["REGIAO"]):
    fig_dashboard.add_trace(
        go.Scatterpolar(
            r=[
                vulnerabilidade_pd.loc[i, "sem_plano"],
                vulnerabilidade_pd.loc[i, "idosos"],
                vulnerabilidade_pd.loc[i, "sintomas_graves"],
                vulnerabilidade_pd.loc[i, "recebe_auxilio"],
                vulnerabilidade_pd.loc[i, "sem_plano"]  # Repetir o primeiro para fechar o polígono
            ],
            theta=["Sem Plano", "Idosos", "Sintomas Graves", "Auxílio", "Sem Plano"],
            name=regiao,
            fill='toself'
        ),
        row=2, col=2
    )

# Atualizar layout do dashboard com mais espaço e ajustes
fig_dashboard.update_layout(
    title_text="Dashboard COVID-19: Principais Indicadores",
    height=1000,  # Aumentar altura
    width=1200,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

# Ajustar layouts específicos para cada subplot
fig_dashboard.update_xaxes(title_text="Faixa Etária", row=1, col=1)
fig_dashboard.update_yaxes(title_text="Percentual (%)", row=1, col=1)

fig_dashboard.update_xaxes(title_text="Nível de Risco", row=2, col=1)
fig_dashboard.update_yaxes(title_text="Região", row=2, col=1)

# Ajustar o gráfico polar para ter mais espaço
fig_dashboard.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    polar_domain=dict(
        x=[0.55, 0.95],  # Ajustar posição horizontal
        y=[0.05, 0.45]   # Ajustar posição vertical
    )
)

# Exibir o dashboard
display(fig_dashboard)

In [0]:
# Salvar os gráficos como HTML para referência futura
html_output = """
<h1>Análise Gráfica PNAD-COVID-19</h1>
<p>Este relatório apresenta os principais resultados da análise dos dados da PNAD-COVID-19.</p>
"""

# Adicionar código para salvar os gráficos como HTML
from IPython.display import HTML

# Exibir mensagem de conclusão
displayHTML("""
<!DOCTYPE html>
<html lang="pt">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Análise Gráfica PNAD-COVID-19</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
            padding: 20px;
            background-color: #f4f4f4;
        }
        .container {
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
        }
        h1, h2 {
            color: #0066cc;
        }
        .info-box {
            background-color: #e6f7ff;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
        }
        ul {
            list-style-type: none;
            padding: 0;
        }
        li {
            margin-bottom: 10px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Análise Gráfica PNAD-COVID-19</h1>
        <p>Este relatório apresenta os principais resultados da análise dos dados da PNAD-COVID-19.</p>
        
        <div class="info-box">
            <h2>Análise Gráfica Concluída</h2>
            <p>Todos os gráficos foram gerados com sucesso a partir dos dados da PNAD-COVID-19.</p>
            <p>Os resultados visuais permitem identificar padrões importantes para a tomada de decisão em caso de novo surto.</p>
        </div>
        
        <div style="background-color:#e6f7ff; padding:20px; border-radius:10px; margin:20px 0;">
            <h2 style="color:#0066cc;">Dashboard Final: Explicação</h2>
            <p>O dashboard final combina quatro visualizações principais em um único painel:</p>
            
            <h3>1. Quadrante Superior Esquerdo: Prevalência de Sintomas Graves por Faixa Etária</h3>
            <ul>
                <li>Gráfico de barras mostrando o percentual de pessoas com sintomas graves em cada faixa etária.</li>
                <li>As barras são coloridas usando a paleta Viridis, com cores mais escuras indicando maior prevalência.</li>
                <li>Permite identificar rapidamente quais grupos etários são mais afetados por sintomas graves.</li>
            </ul>
            
            <h3>2. Quadrante Superior Direito: Distribuição de Auxílio Emergencial</h3>
            <ul>
                <li>Gráfico de pizza mostrando a distribuição percentual do auxílio emergencial por região.</li>
                <li>Cada região é representada por uma cor distinta da paleta Bold.</li>
                <li>Facilita a visualização de quais regiões receberam maior proporção do auxílio.</li>
            </ul>
            
            <h3>3. Quadrante Inferior Esquerdo: Níveis de Risco por Região</h3>
            <ul>
                <li>Mapa de calor mostrando a distribuição de níveis de risco (Baixo, Médio, Alto, Muito Alto) por região.</li>
                <li>Cores mais intensas indicam maior percentual de pessoas naquele nível de risco.</li>
                <li>Permite identificar padrões regionais de vulnerabilidade.</li>
            </ul>
            
            <h3>4. Quadrante Inferior Direito: Perfil de Vulnerabilidade</h3>
            <ul>
                <li>Gráfico de radar mostrando múltiplas dimensões de vulnerabilidade por região.</li>
                <li>Cada região tem seu próprio polígono no gráfico.</li>
                <li>As dimensões incluem: percentual sem plano de saúde, percentual de idosos, percentual com sintomas graves e percentual recebendo auxílio.</li>
                <li>Permite comparação multidimensional entre regiões.</li>
            </ul>
        </div>
    </div>
</body>
</html>

""")